<a href="https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


I am working on **Lane 2: Refresh / Content Opportunity Scoring**. My goal is to rank content pages so that a human reviewer can decide which pages should be reviewed first.

I chose a **Random Forest classifier** because the task involves several observable signals that can interact, such as impressions, clicks, sessions, CTR, average position, content age, freshness, word count, and engagement. A Random Forest can learn non-linear relationships between these signals while remaining easier to inspect than a more complex model.

The target is the defined proxy **`is_declining_label`**, where `1` means the observed `trend_direction` is `"down"`. This is a proxy for the current observed decline and should not be interpreted as a guarantee of future performance or as a causal effect of refreshing a page.

The model will be evaluated as a ranking system using **Precision@50**, because the practical decision is to review a limited number of pages first. The model is useful only if it improves the quality of this review queue compared with my Week-4 baseline.


In [35]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier

print("Libraries loaded successfully.")
print("Method: Random Forest")
print("Target: is_declining_label")
print("Metric: Precision@50")

Libraries loaded successfully.
Method: Random Forest
Target: is_declining_label
Metric: Precision@50


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


I use a **client-grouped split** so that pages belonging to the same client do not appear in both the training and test sets.

This is more honest for this problem because pages from the same client can have similar characteristics. If the same client appeared in both sets, the model could benefit from client-specific patterns rather than demonstrating that it generalizes to unseen clients.

I use 80% of the clients for training and 20% for testing. The test clients are kept completely separate from model training.

The split is performed before model fitting, and the test set is used only for the final comparison with the Week-4 baseline.


In [36]:
from pathlib import Path

repo_path = Path("/content/Anshika-FlyRank-ML")

if not repo_path.exists():
    !git clone -q https://github.com/Anshikaag-28/Anshika-FlyRank-ML.git /content/Anshika-FlyRank-ML

%cd /content/Anshika-FlyRank-ML

data_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

/content/Anshika-FlyRank-ML
Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [37]:
# Create the proxy target defined in the earlier task framing

df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Target created successfully.")
print(df["is_declining_label"].value_counts())

Target created successfully.
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [38]:
feature_candidates = [
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "freshness_days",
    "ctr",
    "avg_position",
    "word_count",
    "engagement_rate"
]

features = [c for c in feature_candidates if c in df.columns]

print("Features being used:")
print(features)

Features being used:
['impressions_90d', 'sessions_90d', 'content_age_days', 'ctr', 'avg_position', 'word_count', 'engagement_rate']


In [39]:
from sklearn.model_selection import train_test_split

# Check that the correct client identifier exists
assert "client_id" in df.columns

# Keep rows with required information
model_df = df.dropna(
    subset=features + ["is_declining_label", "client_id"]
).copy()

# Get unique clients
clients = model_df["client_id"].unique()

# Split CLIENTS, not individual rows
from sklearn.model_selection import train_test_split

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

# Create train and test datasets
train_df = model_df[
    model_df["client_id"].isin(train_clients)
].copy()

test_df = model_df[
    model_df["client_id"].isin(test_clients)
].copy()

print("Total clients:", len(clients))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

Total clients: 32
Training clients: 25
Test clients: 7
Training rows: 19489
Test rows: 2812


In [40]:
# Check that no client appears in both datasets

overlap = set(train_df["client_id"]).intersection(
    set(test_df["client_id"])
)

print("Client overlap:", len(overlap))

assert len(overlap) == 0

print("PASS: No client appears in both train and test.")

Client overlap: 0
PASS: No client appears in both train and test.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


I train the Random Forest using only the observable features defined above. The model produces a probability that each page belongs to the positive proxy group.

For evaluation, I rank the test pages by the model probability and calculate Precision@50.

I compare this with my Week-4 baseline using the **same test population and the same Precision@50 definition**. This makes the comparison meaningful because both approaches are evaluated on the same pages.

The purpose is not to reward the more complex model automatically. The Random Forest should only be considered useful if it provides better ranking quality than the transparent baseline.


In [41]:
# Prepare train and test data

X_train = train_df[features].copy()
y_train = train_df["is_declining_label"].copy()

X_test = test_df[features].copy()
y_test = test_df["is_declining_label"].copy()

# Fill missing numeric values using training medians
train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (19489, 7)
X_test shape: (2812, 7)


In [42]:
# Train Random Forest

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [43]:
def precision_at_k(y_true, scores, k=50):
    """
    Precision@K:
    Number of positive pages among the top K ranked pages / K
    """
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    ranking = np.argsort(scores)[::-1][:k]

    return y_true[ranking].mean()


# Model probabilities
model_scores = rf_model.predict_proba(X_test)[:, 1]

model_precision_50 = precision_at_k(
    y_test,
    model_scores,
    k=50
)

print(f"Random Forest Precision@50: {model_precision_50:.3f}")

Random Forest Precision@50: 0.840


In [44]:
# Recreate the Week-4 transparent baseline on the test set

baseline_test = test_df.copy()

# Create position buckets
baseline_test["position_bucket"] = pd.cut(
    baseline_test["avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["Top 3", "Page 1", "Page 2", "Deep"],
    include_lowest=True
)

# Calculate expected CTR from the TRAINING data only.
# This avoids using test information to construct the benchmark.

baseline_train = train_df.copy()

baseline_train["position_bucket"] = pd.cut(
    baseline_train["avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["Top 3", "Page 1", "Page 2", "Deep"],
    include_lowest=True
)

expected_ctr = (
    baseline_train
    .groupby("position_bucket", observed=True)["ctr"]
    .median()
)

baseline_test["expected_ctr"] = (
    baseline_test["position_bucket"]
    .map(expected_ctr)
)

baseline_test["expected_ctr"] = pd.to_numeric(
    baseline_test["expected_ctr"],
    errors="coerce"
)

baseline_test["ctr"] = pd.to_numeric(
    baseline_test["ctr"],
    errors="coerce"
)

baseline_test["impressions_90d"] = pd.to_numeric(
    baseline_test["impressions_90d"],
    errors="coerce"
)

baseline_test["ctr_gap"] = (
    baseline_test["expected_ctr"] -
    baseline_test["ctr"]
)

baseline_test["baseline_score"] = (
    baseline_test["ctr_gap"].clip(lower=0) *
    np.log1p(baseline_test["impressions_90d"])
)

baseline_test["baseline_score"] = (
    baseline_test["baseline_score"]
    .fillna(0)
)

baseline_precision_50 = precision_at_k(
    baseline_test["is_declining_label"],
    baseline_test["baseline_score"],
    k=50
)

print(f"Week-4 baseline Precision@50: {baseline_precision_50:.3f}")

Week-4 baseline Precision@50: 0.700


In [45]:
comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_precision_50,
        model_precision_50
    ]
})

comparison

,Method,Precision@50
0,Week-4 baseline,0.70
1,Random Forest,0.84


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The main errors are false positives and false negatives in the ranked review queue. A false positive is a page that the model ranks highly even though its proxy label is not positive. A false negative is a positive page that the model ranks too low.

I inspect the top-ranked test pages and compare their observed signals with their proxy labels. This helps identify cases where high impressions, CTR, position, or sessions lead the model to rank a page highly even though the proxy label does not match.

The feature importance results are treated as interpretation rather than causation. A feature being important means that the model relied on it when making predictions; it does not mean that the feature causes a page to decline.

The model is therefore a **decision-support ranking tool**, not a guarantee that a page should be refreshed or that a refresh will improve performance.


In [46]:
# Create test predictions

error_df = test_df[
    ["client_id", "content_id", "is_declining_label"] +
    features
].copy()

error_df["model_probability"] = model_scores

error_df["predicted_positive"] = (
    error_df["model_probability"] >= 0.50
).astype(int)

# False positives
false_positives = error_df[
    (error_df["predicted_positive"] == 1) &
    (error_df["is_declining_label"] == 0)
].copy()

# False negatives
false_negatives = error_df[
    (error_df["predicted_positive"] == 0) &
    (error_df["is_declining_label"] == 1)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 594
False negatives: 398


In [47]:
top_20 = (
    error_df
    .sort_values("model_probability", ascending=False)
    .head(20)
)

display(
    top_20[
        [
            "client_id",
            "content_id",
            "model_probability",
            "is_declining_label"
        ] + features
    ]
)

,client_id,content_id,model_probability,is_declining_label,impressions_90d,sessions_90d,content_age_days,ctr,avg_position,word_count,engagement_rate
25502,client_349c41201b,content_569e0d485db8,0.812022,1,1816,10,154,0.06,28.1,4401.0,0.00
26700,client_349c41201b,content_8d387d58f71f,0.810363,1,1032,16,148,0.00,5.7,6583.0,0.00
2610,client_349c41201b,content_a8cd736c9c7b,0.809200,1,2877,4,144,0.03,23.2,4448.0,0.00
12708,client_349c41201b,content_c14a055e6a26,0.808771,1,8605,30,140,0.07,34.0,4148.0,0.00
13141,client_349c41201b,content_be1c109fcdbc,0.807584,1,10931,19,144,0.05,34.0,5359.0,0.00
27740,client_19581e27de,content_cdc1d5c2b05a,0.806394,1,3989,40,153,0.00,17.0,2910.0,2.50
28130,client_349c41201b,content_0bc3bda5785c,0.805916,0,2929,5,167,0.07,16.9,3420.0,0.00
7570,client_19581e27de,content_a451a30f3920,0.803025,1,11926,13,139,0.04,4.4,4028.0,7.69
2790,client_19581e27de,content_39dbf0817270,0.802735,1,3755,2,98,0.05,1.3,2533.0,0.00
17661,client_349c41201b,content_7ee674751acf,0.802707,1,526,13,168,0.00,21.3,4053.0,0.00


In [48]:
importance = pd.DataFrame({
    "feature": features,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance)

,feature,importance
0,impressions_90d,0.335925
4,avg_position,0.188229
2,content_age_days,0.157384
5,word_count,0.131654
3,ctr,0.082062
1,sessions_90d,0.068703
6,engagement_rate,0.036043


In [49]:
print("Most important features according to the Random Forest:")

for _, row in importance.head(5).iterrows():
    print(
        f"- {row['feature']}: "
        f"{row['importance']:.3f}"
    )

Most important features according to the Random Forest:
- impressions_90d: 0.336
- avg_position: 0.188
- content_age_days: 0.157
- word_count: 0.132
- ctr: 0.082


In [50]:
forbidden_features = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "priority_score",
    "action_type",
    "health_score"
}

leaked_features = set(features).intersection(
    forbidden_features
)

print("Forbidden features found:", leaked_features)

assert len(leaked_features) == 0

print("PASS: No target/product-decision fields are used as model features.")

Forbidden features found: set()
PASS: No target/product-decision fields are used as model features.


In [51]:
assert set(train_df["client_id"]).isdisjoint(
    set(test_df["client_id"])
)

print("PASS: Client-grouped split is leakage-safe.")

PASS: Client-grouped split is leakage-safe.


### Final interpretation

On the held-out test clients, the Random Forest achieved a Precision@50 of 0.84, compared with 0.70 for the Week-4 baseline. The model therefore improved the top-50 ranking quality on this test split. This result is measured on the selected anonymized dataset and should be treated as directional evidence rather than a general benchmark. The model combines multiple observable signals instead of relying on one fixed rule. However, the errors show that a high model score does not guarantee that a page needs a refresh. The output should therefore be used as decision support for human review, not as an automatic action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.